<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/CH09/CH09_NB03_HandsOn_3Expert_MoE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/CH09/CH09_NB03_HandsOn_3Expert_MoE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%; border:none; background:none;">
  <tr style="border:none;">
    <td style="border:none; vertical-align:middle; text-align:left; width: 120px;">
      <a href="https://hubs.la/Q040tvsK0"><img src="https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/Images/cover.png" width="100px" style="border-radius: 4px;"></a>
    </td>
    <td style="border:none; vertical-align:middle; text-align:left;">
      <p style="margin: 0; font-size: 14px;">
        Supplementary code for the <a href="https://hubs.la/Q040tvsK0">Rearchitecting LLMs</a> book by <a href="https://github.com/peremartra">Pere Martra</a>.<br>
        <br>
        Code repository: <a href="https://github.com/peremartra/Rearchitecting-LLMs">https://github.com/peremartra/Rearchitecting-LLMs</a>
      </p>
    </td>
  </tr>
</table>

# Rearchitecting LLMs
## Structural techniques for efficient models

### Chapter 9: Mixture of Experts (MoE)
### Notebook 3: Hands-on Lab — Three-Expert MoE

[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pere-martra/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=flat&logo=github&logoColor=white)](https://github.com/peremartra) [![X](https://img.shields.io/badge/X-000000?style=flat&logo=x&logoColor=white)](https://x.com/PereMartra) [![Hugging Face](https://img.shields.io/badge/🤗%20Hugging%20Face-blue)](https://huggingface.co/oopere)

_____
Colab Environment: GPU T4

Models:
- HuggingFaceTB/SmolLM2-1.7B-Instruct
- oopere/SmolLM2-1.7B-ClinicalNER

_____
In Notebooks 1 and 2 you worked with a two-expert MoE. In this lab you extend
that design to three experts: Expert 0 retains general knowledge from the base
model, Expert 1 brings the clinical specialization learned in Chapter 7, and
Expert 2 is initialised from a copy of the base MLP and trained on a new domain.

Your goal is to train the router alongside Expert 2 and verify, through routing
analysis, that the model has learned to direct each type of input to the right expert.

## Setting up the notebook

This section installs dependencies, imports modules, and defines utility and
configuration values used throughout the notebook.

In [1]:
!pip install -q \
    transformers==5.0.0 \
    datasets==4.0.0 \
    accelerate==1.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 25.9 MB/s eta 0:00:00


In [2]:
import copy
import json
import random
import gc
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

Running on: cuda


In [3]:
def set_seed(seed=42):
    """Set random seed for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print("\u2713 Random seed set to 42")

✓ Random seed set to 42


## Configuration

Centralized parameters for reproducible experiments. Adjust values here before
running the workflow.

In [4]:
# Core model configuration
BASE_MODEL_ID     = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
CLINICAL_MODEL_ID = "oopere/SmolLM2-1.7B-ClinicalNER"

# Runtime and training settings
EPOCHS = 1

# Layers where Expert 2 is unfrozen during training.
# Change this range to explore how the depth of adaptation
# affects routing behaviour and Expert 2 specialization.
EXPERT1_TRAINABLE_LAYERS = range(21, 24)
EXPERT2_TRAINABLE_LAYERS = range(21, 24)

# Samples per domain used during training
TRAIN_SAMPLES_PER_DOMAIN = 300

# Sequence and generation settings
MAX_TRAIN_LENGTH         = 512
MAX_GENERAL_TEST_TOKENS  = 50
MAX_CODE_TEST_TOKENS     = 150
MAX_CLINICAL_TEST_TOKENS = 150
MAX_EVAL_TOKENS          = 400

# Reproducibility
SEED = 42
set_seed(SEED)
print(f"Configuration loaded | seed={SEED} | epochs={EPOCHS} | expert2 layers={list(EXPERT2_TRAINABLE_LAYERS)}")

Configuration loaded | seed=42 | epochs=1 | expert2 layers=[21, 22, 23]


In [5]:
def clear_gpu_cache():
    """Clear GPU cache completely"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()

## Workflow: Building the three-expert MoE

### Loading the models

We need both models in memory only for a short phase. After extracting the clinical
MLP weights from the Chapter 7 model and injecting them as Expert 1, we release
that model before training. Expert 2 is initialised automatically from a copy
of the base MLP by the class constructor.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# The base model becomes Expert 0 (general knowledge).
print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID,
                                             torch_dtype=torch.float16)
model.to(DEVICE)

print(f"\nBase model     : {sum(p.numel() for p in model.parameters()):,} parameters")

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

Loading base model...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


Base model     : 1,711,376,384 parameters


## The MoE block

The class is identical to Notebooks 1 and 2. Expert 2 does not need an explicit
argument — the `while` loop inside `__init__` fills any remaining slot with a copy
of the original MLP automatically.

That single filling mechanism is enough to go from two experts to three: the only
change at the call site is `num_experts=3`.

In [7]:
class TrainableTopKMoE(nn.Module):
    """Top-k soft/hard MoE block with a configurable number of experts.

    Args:
        original_mlp:  the original MLP from the decoder layer (becomes Expert 0).
        hidden_size:   model hidden dimension, used to size the router.
        num_experts:   total number of experts in this block.
        top_k:         number of experts considered per token during inference
                       (must be <= num_experts). Mutable after construction —
                       e.g. `layer.top_k = 1` to switch behaviour without rebuilding.
        extra_experts: optional list of pre-trained MLPs to use as Expert 1..N.
                       Missing slots (if len(extra_experts) < num_experts - 1)
                       are filled with copies of original_mlp, same fallback
                       behaviour as before.
    """
    def __init__(self, original_mlp, hidden_size, num_experts=2, top_k=2,
                 extra_experts=None):
        super().__init__()
        assert top_k <= num_experts, "top_k must be lower than number of experts"

        target_dtype  = next(original_mlp.parameters()).dtype
        target_device = next(original_mlp.parameters()).device

        self.num_experts = num_experts
        self.top_k = top_k

        experts_list = [original_mlp]
        extra_experts = extra_experts or []
        for e in extra_experts:
            experts_list.append(
                copy.deepcopy(e).to(device=target_device, dtype=target_dtype)
            )
        # Fill any remaining slots with copies of the original MLP.
        while len(experts_list) < num_experts:
            experts_list.append(
                copy.deepcopy(original_mlp).to(device=target_device, dtype=target_dtype)
            )

        self.experts = nn.ModuleList(experts_list)
        self.router = nn.Linear(hidden_size, num_experts, bias=False,
                                dtype=target_dtype, device=target_device)
        self.last_routing = None

    def forward(self, hidden_states):
        router_logits = self.router(hidden_states)

        if self.training:
            # Dense soft routing — gradients flow through the weighted sum
            # over ALL experts. top_k is not used here.
            routing_weights = F.softmax(
                router_logits.float(), dim=-1).to(hidden_states.dtype)

            output = torch.zeros_like(hidden_states)
            for i in range(self.num_experts):
                output = output + self.experts[i](hidden_states) * \
                    routing_weights[..., i].unsqueeze(-1)

            self.last_routing = routing_weights.detach()

        else:
            # Sparse top-k routing — only the selected experts run.
            topk_values, topk_indices = torch.topk(
                router_logits, k=self.top_k, dim=-1)
            # Softmax renormalised over just the selected experts.
            topk_weights = F.softmax(
                topk_values.float(), dim=-1).to(hidden_states.dtype)

            output = torch.zeros_like(hidden_states)
            full_weights = torch.zeros_like(router_logits)

            for expert_idx in range(self.num_experts):
                # Mask: which tokens picked this expert, in any top-k slot.
                is_selected = (topk_indices == expert_idx)
                if not is_selected.any():
                    continue
                token_mask = is_selected.any(dim=-1)
                weight_per_token = (topk_weights * is_selected).sum(dim=-1)

                expert_out = self.experts[expert_idx](hidden_states[token_mask])
                output[token_mask] += expert_out * \
                    weight_per_token[token_mask].unsqueeze(-1)
                full_weights[..., expert_idx] = weight_per_token

            self.last_routing = full_weights.detach()

        return output

## Surgery

We extract the clinical MLP from the Chapter 7 model and inject it as Expert 1
in every layer. Expert 2 is filled automatically by the class constructor from a
copy of the base MLP. Both Expert 0 and Expert 2 start from the same base weights;
the router will learn to distinguish them as Expert 2 adapts to the code domain
during training.

In [8]:
LAYERS_TO_MODIFY = range(model.config.num_hidden_layers)

In [9]:
# Extract only the MLP weights from the chapter-7 model — CPU only.
hidden_size = model.config.hidden_size
print("Extracting clinical MLPs from chapter-7 model...")
clinical_mlps = {}
clinical_model = AutoModelForCausalLM.from_pretrained(
    CLINICAL_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu"
)
for layer_idx in LAYERS_TO_MODIFY:
    clinical_mlps[layer_idx] = copy.deepcopy(
        clinical_model.model.layers[layer_idx].mlp
    )

# Chapter-7 model has served its purpose — free the memory before surgery.
del clinical_model
torch.cuda.empty_cache()
print("Clinical model unloaded.")

# Now perform the surgery with only one model in GPU memory.
# num_experts=3: Expert 0 = base MLP (general knowledge),
#                Expert 1 = clinical MLP from Chapter 7,
#                Expert 2 = copy of base MLP (filled automatically).
for layer_idx in LAYERS_TO_MODIFY:
    base_mlp = model.model.layers[layer_idx].mlp
    model.model.layers[layer_idx].mlp = TrainableTopKMoE(
        base_mlp, hidden_size, num_experts=3, top_k=2,
        extra_experts=[clinical_mlps[layer_idx]]
    )

del clinical_mlps
torch.cuda.empty_cache()
print(f"Transplant complete: {len(LAYERS_TO_MODIFY)} layers | 3 experts per block.")

Extracting clinical MLPs from chapter-7 model...


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Clinical model unloaded.
Transplant complete: 24 layers | 3 experts per block.


In [10]:
gc.collect()

252

## Freezing strategy

Expert 0 and Expert 1 are fully frozen. Expert 2 is frozen everywhere except the
tail layers defined by `EXPERT2_TRAINABLE_LAYERS`. The router is always trainable
across all 24 layers.

This mirrors the Phase 2 logic from Notebook 2: the tail layers of Expert 2 are
where the final token-level representations are shaped, and adapting them is enough
to push the expert toward a new domain without retraining from scratch.

In [11]:
# Freeze the entire model.
for param in model.parameters():
    param.requires_grad_(False)

# Unfreeze the routers — one per MoE block, across all layers.
for layer_idx in LAYERS_TO_MODIFY:
    for param in model.model.layers[layer_idx].mlp.router.parameters():
        param.requires_grad_(True)

# Unfreeze Expert 1 and Expert 2 tail layers.
# Both experts adapt their final representations jointly with the router.
for layer_idx in EXPERT1_TRAINABLE_LAYERS:
    for param in model.model.layers[layer_idx].mlp.experts[1].parameters():
        param.requires_grad_(True)

for layer_idx in EXPERT2_TRAINABLE_LAYERS:
    for param in model.model.layers[layer_idx].mlp.experts[2].parameters():
        param.requires_grad_(True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Total parameters                              : {total:,}")
print(f"Trainable (router + E1 tail + E2 tail)        : {trainable:,} ({trainable/total:.4%})")

Total parameters                              : 4,127,442,944
Trainable (router + E1 tail + E2 tail)        : 302,137,344 (7.3202%)


## Dataset preparation

The router must see all three domains during training to learn the routing policy.
We load one dataset per domain, take the same number of samples from each, and
shuffle them into a single training list.

| Domain   | Dataset                        | Format              |
|----------|--------------------------------|---------------------|
| Clinical | oopere/clinical-ner-qdora      | NER extraction      |
| General  | HuggingFaceTB/smoltalk         | Single-turn chat    |
| Code     | flytech/python-codes-25k       | Python code snippet |

In [12]:
# --- Clinical domain ---
dataset = load_dataset("oopere/clinical-ner-qdora", split="train")
train_dataset = dataset.select(range(TRAIN_SAMPLES_PER_DOMAIN))

clinical_prompts = []
for item in train_dataset:
    label = json.loads(item['label']) if isinstance(item['label'], str) else item['label']
    messages = [
        {"role": "system",    "content": "Extract:"},
        {"role": "user",      "content": item['note']},
        {"role": "assistant", "content": json.dumps(label, indent=2)}
    ]
    clinical_prompts.append(
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    )

# --- General domain ---
smoltalk = load_dataset("HuggingFaceTB/smoltalk", "all", split="train")
smoltalk_single = smoltalk.filter(
    lambda x: len(x["messages"]) == 2 and
              len(x["messages"][0]["content"]) < 300
)
general_subset = smoltalk_single.select(range(TRAIN_SAMPLES_PER_DOMAIN))

general_prompts = []
for item in general_subset:
    general_prompts.append(
        tokenizer.apply_chat_template(
            item["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    )

# --- Code domain ---
code_dataset = load_dataset("flytech/python-codes-25k", split="train")
code_subset = code_dataset.select(range(TRAIN_SAMPLES_PER_DOMAIN))

code_prompts = []
for item in code_subset:
    messages = [
        {"role": "user",      "content": "Write Python code:"},
        {"role": "assistant", "content": item['output']}
    ]
    code_prompts.append(
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    )

# --- Mix and shuffle ---
training_prompts = clinical_prompts + general_prompts + code_prompts
random.shuffle(training_prompts)

print(f"Training prompts — clinical: {len(clinical_prompts)} | "
      f"general: {len(general_prompts)} | "
      f"code: {len(code_prompts)} | "
      f"total: {len(training_prompts)}")

README.md:   0%|          | 0.00/4.90k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/102k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/360 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/40 [00:00<?, ? examples/s]

...

README.md:   0%|          | 0.00/3.29k [00:00<?, ?B/s]

python-codes-25k.json:   0%|          | 0.00/26.4M [00:00<?, ?B/s]

python-codes-25k.jsonl:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49626 [00:00<?, ? examples/s]

Training prompts — clinical: 300 | general: 300 | code: 300 | total: 900


## Training

We train the router and the Expert 2 tail layers jointly. Two learning rates keep
the dynamics balanced: the router uses a higher rate to converge quickly on the
routing policy; Expert 2 uses a lower one to specialise gradually without losing
its initialisation.

In [13]:
clear_gpu_cache()

def train_moe(model):
    model.config.use_cache = False
    model.to(DEVICE).train()

    # Promote only trainable parameters to FP32 for numerical stability.
    for param in model.parameters():
        if param.requires_grad:
            param.data = param.data.float()

    # ---------------------------------------------------------------
    # OPTIONAL: Gaussian noise on router logits
    # To activate noise-based exploration, add the following line
    # inside TrainableTopKMoE.forward(), right after computing
    # router_logits and inside the `if self.training:` block:
    #
    #   router_logits = router_logits + torch.randn_like(router_logits) * 0.1
    #
    # Expert 0 and Expert 2 start as near-identical copies of the same
    # base MLP. Without noise, their logits are nearly equal and the
    # gradient signal pushing the router to distinguish them is weak
    # in early steps. Noise breaks that symmetry and encourages the
    # router to explore both experts before Expert 2 has had time to
    # diverge on its own.
    # Activate this only if Expert 2 receives near-zero routing weight
    # after the first epoch.
    # ---------------------------------------------------------------

    # Two parameter groups: router converges fast; Expert 2 adapts slowly.
    router_params  = []
    expert2_params = []

    for layer_idx in LAYERS_TO_MODIFY:
        router_params.extend(
            model.model.layers[layer_idx].mlp.router.parameters()
        )
    for layer_idx in EXPERT2_TRAINABLE_LAYERS:
        expert2_params.extend(
            model.model.layers[layer_idx].mlp.experts[2].parameters()
        )

    optimizer = optim.AdamW([
        {"params": router_params,  "lr": 1e-5},
        {"params": expert2_params, "lr": 1e-4},
    ])

    scaler = GradScaler('cuda')

    for epoch in range(EPOCHS):
        total_loss = 0
        random.shuffle(training_prompts)

        for idx, text in enumerate(training_prompts):
            inputs = tokenizer(
                text, return_tensors="pt", truncation=True,
                max_length=MAX_TRAIN_LENGTH
            ).to(DEVICE)

            optimizer.zero_grad()

            with autocast('cuda', dtype=torch.float16):
                loss = model(**inputs, labels=inputs["input_ids"]).loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

            if (idx + 1) % 100 == 0:
                print(f"  Epoch {epoch+1} step {idx+1}/{len(training_prompts)} "
                      f"loss: {loss.item():.4f}")

        print(f"-> END EPOCH {epoch+1:02d} avg loss: {total_loss/len(training_prompts):.4f}")

    model.config.use_cache = True
    print("Training complete.")

In [14]:
gc.collect()
torch.cuda.empty_cache()
train_moe(model)

  Epoch 1 step 100/900 loss: 0.8764
  Epoch 1 step 200/900 loss: 0.3389
  Epoch 1 step 300/900 loss: 0.6507
  Epoch 1 step 400/900 loss: 0.3146
  Epoch 1 step 500/900 loss: 0.4692
  Epoch 1 step 600/900 loss: 0.6046
  Epoch 1 step 700/900 loss: 0.8004
  Epoch 1 step 800/900 loss: 0.4575
  Epoch 1 step 900/900 loss: 0.2418
-> END EPOCH 01 avg loss: 0.6093
Training complete.


## Inference control

We test the model on one prompt per domain. The goal is not to measure accuracy
but to verify that the model responds coherently to each type of input after training.

Each prompt is run twice: with `top_k=2` (the two highest-scoring experts blend their
outputs) and with `top_k=1` (only the winning expert runs).

In [15]:
def run_moe(prompt, max_tokens=400):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)


def set_top_k(model, k):
    """Set top_k on every MoE layer of the model.

    A global inference-time switch: k=num_experts reproduces the dense
    soft blend used during training; k=1 hard-routes each token to its
    highest-scoring expert.
    """
    for layer in model.model.layers:
        if hasattr(layer.mlp, "top_k"):
            layer.mlp.top_k = k

In [16]:
model.to(torch.float16)
model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): TrainableTopKMoE(
          (experts): ModuleList(
            (0-2): 3 x LlamaMLP(
              (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
              (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
              (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
              (act_fn): SiLUActivation()
            )
          )
          (router): Linear(in_features=2048, out_features=3, bias=False)


In [17]:
# Build prompts once — reused across top_k=2 and top_k=1 runs.
messages_general = [{"role": "user", "content": "What's the most significant building in Barcelona?"}]
prompt_general = tokenizer.apply_chat_template(
    messages_general, tokenize=False, add_generation_prompt=True)

nota_test = (
    "Patient is a 50 yo F c/o severe chest pain and SOB since yesterday. "
    "Vitals: HR 110, BP 150/90. Started on Aspirin 81mg."
)
messages_clinical = [
    {"role": "system", "content": "Extract:"},
    {"role": "user",   "content": nota_test}
]
prompt_clinical = tokenizer.apply_chat_template(
    messages_clinical, tokenize=False, add_generation_prompt=True)

messages_code = [{"role": "user", "content": "Explain this line: copy.deepcopy(e).to(device=target_device, dtype=target_dtype)"}]
prompt_code = tokenizer.apply_chat_template(
    messages_code, tokenize=False, add_generation_prompt=True)

### top_k = 2 — soft blend (two experts contribute)

In [18]:
set_top_k(model, 2)
print("=" * 56)
print(" INFERENCE CONTROL — top_k = 2 (soft blend)")
print("=" * 56)

print("\n[Test A — General knowledge]")
print(f"Response: {run_moe(prompt_general, max_tokens=MAX_GENERAL_TEST_TOKENS)}")

print("\n[Test B — Clinical NER]")
print(f"Response: {run_moe(prompt_clinical, max_tokens=MAX_CLINICAL_TEST_TOKENS)}")

print("\n[Test C — Code explanation]")
print(f"Response: {run_moe(prompt_code, max_tokens=MAX_CODE_TEST_TOKENS)}")

 INFERENCE CONTROL — top_k = 2 (soft blend)

[Test A — General knowledge]
Response: The most significant building in Barcelona is the Sagrada Familia. Completed by Antoni Gaudí, it is a large, ornate church that is a major tourist attraction.

[Test B — Clinical NER]
Response: {
  "patient_age": 50,
  "symptoms": [
    "chest pain",
    "shortness of breath"
  ],
  "vital_signs": {
    "temperature": null,
    "heart_rate": 110,
    "blood_pressure": "150/90"
  },
  "medications": [
    {
      "name": "Aspirin",
      "dose": "81mg",
      "frequency": "not specified"
    }
  ],
  "duration_days": 1
}

[Test C — Code explanation]
Response: ```python
copy.deepcopy(e).to(device=target_device, dtype=target_dtype)
```

This line of code is used to create a deep copy of the tensor `e` and then move it to the target device `target_device` with the target data type `target_dtype`.

Here's a breakdown of what each part does:

- `copy.deepcopy(e)`: This function creates a deep copy of the tens

### top_k = 1 — hard routing (single expert per token)

In [19]:
set_top_k(model, 1)
print("=" * 56)
print(" INFERENCE CONTROL — top_k = 1 (hard routing)")
print("=" * 56)

print("\n[Test A — General knowledge]")
print(f"Response: {run_moe(prompt_general, max_tokens=MAX_GENERAL_TEST_TOKENS)}")

print("\n[Test B — Clinical NER]")
print(f"Response: {run_moe(prompt_clinical, max_tokens=MAX_CLINICAL_TEST_TOKENS)}")

print("\n[Test C — Code explanation]")
print(f"Response: {run_moe(prompt_code, max_tokens=MAX_CODE_TEST_TOKENS)}")

 INFERENCE CONTROL — top_k = 1 (hard routing)

[Test A — General knowledge]
Response: The most significant building in Barcelona is the Sagrada Familia, a large-scale Roman Catholic church designed by Antoni Gaudí.

[Test B — Clinical NER]
Response: {
  "patient_age": 50
}

[Test C — Code explanation]
Response: ```python
copy.deepcopy(e).to(device=target_device, dtype=target_dtype)
```
**Explanation:**
```python
```


## Routing analysis

We inspect token-level routing probabilities in layers 18 and 23 for each of the
three prompts. Layer 18 is the first layer where Expert 2 was allowed to train;
layer 23 is the last layer of the model.

A well-trained router should show Expert 1 dominating on clinical tokens,
Expert 2 capturing code tokens, and a more distributed pattern for general text.

In [20]:
def analyze_routing(text, layer_index=0):
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=torch.float16):
            _ = model(**inputs)
    moe_layer = model.model.layers[layer_index].mlp
    routing_weights = moe_layer.last_routing.squeeze(0)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    num_experts = moe_layer.num_experts
    header = " | ".join(f"Expert {i}" for i in range(num_experts))
    print(f"Routing analysis (layer {layer_index}, top_k={moe_layer.top_k})")
    print(f"{'Token':<18} | {header}")
    print("-" * (21 + 12 * num_experts))
    for token, weights in zip(tokens, routing_weights):
        clean = token.replace("\u0120", "").replace(" ", "")
        weight_str = " | ".join(
            f"{weights[i].item():<10.4f}" for i in range(num_experts)
        )
        print(f"{clean:<18} | {weight_str}")

### top_k = 2 — soft blend

In [21]:
set_top_k(model, 2)

#### Layer 18

In [22]:
analyze_routing("What's the most significant building in Barcelona?", layer_index=18)

Routing analysis (layer 18, top_k=2)
Token              | Expert 0 | Expert 1 | Expert 2
---------------------------------------------------------
What               | 0.5010     | 0.0000     | 0.4990    
's                 | 0.4929     | 0.5068     | 0.0000    
the                | 0.4653     | 0.5347     | 0.0000    
most               | 0.5112     | 0.4890     | 0.0000    
significant        | 0.4634     | 0.5366     | 0.0000    
building           | 0.5107     | 0.0000     | 0.4893    
in                 | 0.4905     | 0.0000     | 0.5093    
Barcelona          | 0.0000     | 0.4998     | 0.5000    
?                  | 0.5024     | 0.4976     | 0.0000    


In [23]:
analyze_routing("Patient is a 67-year-old male with chronic pain", layer_index=18)

Routing analysis (layer 18, top_k=2)
Token              | Expert 0 | Expert 1 | Expert 2
---------------------------------------------------------
Patient            | 0.5005     | 0.0000     | 0.4993    
is                 | 0.4636     | 0.5366     | 0.0000    
a                  | 0.4346     | 0.5654     | 0.0000    
                   | 0.4453     | 0.5547     | 0.0000    
6                  | 0.4036     | 0.5962     | 0.0000    
7                  | 0.4392     | 0.5605     | 0.0000    
-                  | 0.4336     | 0.5664     | 0.0000    
year               | 0.4685     | 0.5317     | 0.0000    
-                  | 0.0000     | 0.5547     | 0.4451    
old                | 0.4287     | 0.5713     | 0.0000    
male               | 0.4004     | 0.5996     | 0.0000    
with               | 0.3652     | 0.6348     | 0.0000    
chronic            | 0.4250     | 0.5747     | 0.0000    
pain               | 0.3833     | 0.6167     | 0.0000    


In [24]:
analyze_routing("Explain this line: copy.deepcopy(e).to(device=target_device, dtype=target_dtype)", layer_index=18)

Routing analysis (layer 18, top_k=2)
Token              | Expert 0 | Expert 1 | Expert 2
---------------------------------------------------------
Explain            | 0.5010     | 0.0000     | 0.4988    
this               | 0.4954     | 0.5044     | 0.0000    
line               | 0.4841     | 0.5161     | 0.0000    
:                  | 0.4956     | 0.5044     | 0.0000    
copy               | 0.4790     | 0.5210     | 0.0000    
.                  | 0.4612     | 0.5386     | 0.0000    
deepcopy           | 0.4578     | 0.5425     | 0.0000    
(                  | 0.0000     | 0.4983     | 0.5020    
e                  | 0.0000     | 0.4890     | 0.5107    
).                 | 0.0000     | 0.5249     | 0.4749    
to                 | 0.4866     | 0.5137     | 0.0000    
(                  | 0.4580     | 0.5420     | 0.0000    
device             | 0.0000     | 0.5273     | 0.4729    
=                  | 0.0000     | 0.5239     | 0.4763    
target             | 0.0000     | 0.4961 

#### Layer 23

In [25]:
analyze_routing("What's the most significant building in Barcelona?", layer_index=23)

Routing analysis (layer 23, top_k=2)
Token              | Expert 0 | Expert 1 | Expert 2
---------------------------------------------------------
What               | 0.5029     | 0.0000     | 0.4971    
's                 | 0.5005     | 0.4995     | 0.0000    
the                | 0.4897     | 0.0000     | 0.5103    
most               | 0.5176     | 0.4827     | 0.0000    
significant        | 0.5371     | 0.0000     | 0.4629    
building           | 0.5264     | 0.0000     | 0.4736    
in                 | 0.5161     | 0.0000     | 0.4839    
Barcelona          | 0.4980     | 0.0000     | 0.5020    
?                  | 0.4424     | 0.0000     | 0.5576    


In [26]:
analyze_routing("Patient is a 67-year-old male with chronic pain", layer_index=23)

Routing analysis (layer 23, top_k=2)
Token              | Expert 0 | Expert 1 | Expert 2
---------------------------------------------------------
Patient            | 0.5063     | 0.0000     | 0.4934    
is                 | 0.4277     | 0.0000     | 0.5723    
a                  | 0.5142     | 0.0000     | 0.4858    
                   | 0.5142     | 0.0000     | 0.4858    
6                  | 0.5615     | 0.0000     | 0.4387    
7                  | 0.4780     | 0.0000     | 0.5220    
-                  | 0.4736     | 0.0000     | 0.5264    
year               | 0.4812     | 0.0000     | 0.5186    
-                  | 0.4690     | 0.0000     | 0.5312    
old                | 0.4399     | 0.0000     | 0.5601    
male               | 0.4099     | 0.0000     | 0.5898    
with               | 0.4585     | 0.0000     | 0.5415    
chronic            | 0.4893     | 0.0000     | 0.5107    
pain               | 0.4375     | 0.0000     | 0.5625    


In [27]:
analyze_routing("Explain this line: copy.deepcopy(e).to(device=target_device, dtype=target_dtype)", layer_index=23)

Routing analysis (layer 23, top_k=2)
Token              | Expert 0 | Expert 1 | Expert 2
---------------------------------------------------------
Explain            | 0.5098     | 0.0000     | 0.4905    
this               | 0.5273     | 0.0000     | 0.4729    
line               | 0.5112     | 0.4890     | 0.0000    
:                  | 0.4270     | 0.0000     | 0.5732    
copy               | 0.4934     | 0.0000     | 0.5068    
.                  | 0.0000     | 0.4907     | 0.5093    
deepcopy           | 0.5195     | 0.0000     | 0.4805    
(                  | 0.4893     | 0.0000     | 0.5107    
e                  | 0.4885     | 0.0000     | 0.5112    
).                 | 0.4351     | 0.0000     | 0.5649    
to                 | 0.4717     | 0.0000     | 0.5283    
(                  | 0.5742     | 0.0000     | 0.4258    
device             | 0.5024     | 0.0000     | 0.4976    
=                  | 0.5605     | 0.0000     | 0.4397    
target             | 0.5103     | 0.0000 

### top_k = 1 — hard routing

In [28]:
set_top_k(model, 1)

#### Layer 18

In [29]:
analyze_routing("What's the most significant building in Barcelona?", layer_index=18)

Routing analysis (layer 18, top_k=1)
Token              | Expert 0 | Expert 1 | Expert 2
---------------------------------------------------------
What               | 1.0000     | 0.0000     | 0.0000    
's                 | 0.0000     | 1.0000     | 0.0000    
the                | 0.0000     | 1.0000     | 0.0000    
most               | 1.0000     | 0.0000     | 0.0000    
significant        | 0.0000     | 1.0000     | 0.0000    
building           | 1.0000     | 0.0000     | 0.0000    
in                 | 0.0000     | 0.0000     | 1.0000    
Barcelona          | 0.0000     | 1.0000     | 0.0000    
?                  | 1.0000     | 0.0000     | 0.0000    


In [30]:
analyze_routing("Patient is a 67-year-old male with chronic pain", layer_index=18)

Routing analysis (layer 18, top_k=1)
Token              | Expert 0 | Expert 1 | Expert 2
---------------------------------------------------------
Patient            | 1.0000     | 0.0000     | 0.0000    
is                 | 0.0000     | 1.0000     | 0.0000    
a                  | 0.0000     | 1.0000     | 0.0000    
                   | 0.0000     | 1.0000     | 0.0000    
6                  | 0.0000     | 1.0000     | 0.0000    
7                  | 0.0000     | 1.0000     | 0.0000    
-                  | 0.0000     | 1.0000     | 0.0000    
year               | 0.0000     | 1.0000     | 0.0000    
-                  | 0.0000     | 1.0000     | 0.0000    
old                | 0.0000     | 1.0000     | 0.0000    
male               | 0.0000     | 1.0000     | 0.0000    
with               | 0.0000     | 1.0000     | 0.0000    
chronic            | 0.0000     | 1.0000     | 0.0000    
pain               | 0.0000     | 1.0000     | 0.0000    


In [31]:
analyze_routing("Explain this line: copy.deepcopy(e).to(device=target_device, dtype=target_dtype)", layer_index=18)

Routing analysis (layer 18, top_k=1)
Token              | Expert 0 | Expert 1 | Expert 2
---------------------------------------------------------
Explain            | 1.0000     | 0.0000     | 0.0000    
this               | 0.0000     | 1.0000     | 0.0000    
line               | 0.0000     | 1.0000     | 0.0000    
:                  | 0.0000     | 1.0000     | 0.0000    
copy               | 0.0000     | 1.0000     | 0.0000    
.                  | 0.0000     | 1.0000     | 0.0000    
deepcopy           | 0.0000     | 1.0000     | 0.0000    
(                  | 0.0000     | 0.0000     | 1.0000    
e                  | 0.0000     | 0.0000     | 1.0000    
).                 | 0.0000     | 1.0000     | 0.0000    
to                 | 0.0000     | 1.0000     | 0.0000    
(                  | 0.0000     | 1.0000     | 0.0000    
device             | 0.0000     | 1.0000     | 0.0000    
=                  | 0.0000     | 1.0000     | 0.0000    
target             | 0.0000     | 0.0000 

#### Layer 23

In [32]:
analyze_routing("What's the most significant building in Barcelona?", layer_index=23)

Routing analysis (layer 23, top_k=1)
Token              | Expert 0 | Expert 1 | Expert 2
---------------------------------------------------------
What               | 0.0000     | 0.0000     | 1.0000    
's                 | 1.0000     | 0.0000     | 0.0000    
the                | 0.0000     | 0.0000     | 1.0000    
most               | 1.0000     | 0.0000     | 0.0000    
significant        | 1.0000     | 0.0000     | 0.0000    
building           | 1.0000     | 0.0000     | 0.0000    
in                 | 1.0000     | 0.0000     | 0.0000    
Barcelona          | 0.0000     | 0.0000     | 1.0000    
?                  | 0.0000     | 0.0000     | 1.0000    


In [33]:
analyze_routing("Patient is a 67-year-old male with chronic pain", layer_index=23)

Routing analysis (layer 23, top_k=1)
Token              | Expert 0 | Expert 1 | Expert 2
---------------------------------------------------------
Patient            | 0.0000     | 0.0000     | 1.0000    
is                 | 0.0000     | 0.0000     | 1.0000    
a                  | 1.0000     | 0.0000     | 0.0000    
                   | 0.0000     | 0.0000     | 1.0000    
6                  | 1.0000     | 0.0000     | 0.0000    
7                  | 0.0000     | 0.0000     | 1.0000    
-                  | 0.0000     | 0.0000     | 1.0000    
year               | 0.0000     | 0.0000     | 1.0000    
-                  | 0.0000     | 0.0000     | 1.0000    
old                | 0.0000     | 0.0000     | 1.0000    
male               | 0.0000     | 0.0000     | 1.0000    
with               | 0.0000     | 0.0000     | 1.0000    
chronic            | 0.0000     | 0.0000     | 1.0000    
pain               | 0.0000     | 0.0000     | 1.0000    


In [34]:
analyze_routing("Explain this line: copy.deepcopy(e).to(device=target_device, dtype=target_dtype)", layer_index=23)

Routing analysis (layer 23, top_k=1)
Token              | Expert 0 | Expert 1 | Expert 2
---------------------------------------------------------
Explain            | 0.0000     | 0.0000     | 1.0000    
this               | 1.0000     | 0.0000     | 0.0000    
line               | 0.0000     | 0.0000     | 1.0000    
:                  | 0.0000     | 0.0000     | 1.0000    
copy               | 0.0000     | 0.0000     | 1.0000    
.                  | 0.0000     | 0.0000     | 1.0000    
deepcopy           | 1.0000     | 0.0000     | 0.0000    
(                  | 0.0000     | 0.0000     | 1.0000    
e                  | 0.0000     | 0.0000     | 1.0000    
).                 | 0.0000     | 0.0000     | 1.0000    
to                 | 0.0000     | 0.0000     | 1.0000    
(                  | 1.0000     | 0.0000     | 0.0000    
device             | 1.0000     | 0.0000     | 0.0000    
=                  | 1.0000     | 0.0000     | 0.0000    
target             | 0.0000     | 0.0000 